In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import scipy 
import cansig 
import matplotlib.pyplot as plt
import pathlib as pl
import os
from scipy import stats

Global seed set to 0


In [2]:
import benchmark.analysis.cnv as cnv
import benchmark.metrics.cnvmetrics as cnvmetrics
import benchmark.metrics.cellmetrics as cellmetrics
import benchmark.metrics.genemetrics as genemetrics
import benchmark.plot.plot_utils as plot_utils
import benchmark.plot.single_plots as single_plots
from benchmark.plot.comparison_plots import *
import benchmark.base.utils as utils


# 1. Setting Parameters & Configuration

In [3]:
# Set these to the file we want to analyze (need to be in folder /data, with the structure /data/DATASET/FILENAME.h5ad)
DATASET1 = 'simulated_splatter'
FILENAME1 = 'simulated_splatter.h5ad' 
DATASET1_NAME = 'Simulated Splatter'

DATASET2 = 'breast'
FILENAME2 = 'breast.h5ad' 
DATASET2_NAME = 'Breast'

# inferCNV arguments
STEP = 5
# Threshold for maximum distance between two cnv segments for merging into connected region
THRESHOLD = 5000000

# 2. Data Preparation 

In [4]:
path = !pwd
path = path[0]
datapath1 = os.path.join(path, 'data', DATASET1)
datapath2 = os.path.join(path, 'data', DATASET2)
path

'/notebook/Z'

# 3. Comparison of distributions

In [5]:
#for all metrics except location
resultpath1 = os.path.join(datapath1, 'results')
resultpath2 = os.path.join(datapath2, 'results')

resultpath_comp = os.path.join(path, 'data', 'comparison' + '_' + DATASET1 + '_vs_' + DATASET2, 'results')

isExist = os.path.exists(resultpath_comp )
if not isExist:
   os.makedirs(resultpath_comp )

ks_cnv = cnvmetrics.get_ks_cnvmetrics(resultpath1, resultpath2, STEP, THRESHOLD)
ks_cell = cellmetrics.get_ks_cellmetrics(resultpath1, resultpath2)
ks_gene = genemetrics.get_ks_genemetrics(resultpath1, resultpath2)

pd.concat([ks_cnv, ks_cell, ks_gene], ignore_index=True).set_index(keys = 'metrics').to_csv(os.path.join(resultpath_comp, 'KS_distance.csv'))

# 4. Comparison plots

In [6]:
plotpath = os.path.join(path, 'plots', 'comparison' + '_' + DATASET1 + '_vs_' + DATASET2)
resultpath1 = os.path.join(datapath1, 'results')
resultpath2 = os.path.join(datapath2, 'results')
resultpath_comp = os.path.join(path, 'data', 'comparison' + '_' + DATASET1 + '_vs_' + DATASET2, 'results')

isExist = os.path.exists(plotpath)
if not isExist:
   os.makedirs(plotpath)

paramstring = '_step' + str(STEP) + '_thresh' + str(THRESHOLD)

In [7]:
#single comparison plots
df1 = plot_utils.read_data(os.path.join(resultpath1, 'cnv_regions' + paramstring + '.csv'))
df2 = plot_utils.read_data(os.path.join(resultpath2, 'cnv_regions' + paramstring + '.csv'))
plot_cnv_bp(df1, df2, percentile=95, output_dir=plotpath, 
            output_name='cnv_regions_bp' + paramstring, 
            df1_name=DATASET1_NAME, df2_name=DATASET2_NAME)
plot_cnv_gene(df1, df2, percentile=97.5, output_dir=plotpath, 
              output_name='cnv_regions_gene' + paramstring, 
              df1_name=DATASET1_NAME, df2_name=DATASET2_NAME)

df1 = read_data(os.path.join(resultpath1, 'cnvmetrics_per_subclonal' + paramstring + '.csv'))
df2 = read_data(os.path.join(resultpath2, 'cnvmetrics_per_subclonal' + paramstring + '.csv'))
plot_stat_bp(df1, df2, percentile=75.0, output_dir= plotpath, 
             output_name='cnv_stats' + paramstring, 
             df1_name=DATASET1_NAME, df2_name=DATASET2_NAME)
plot_nregions_per_cell(df1, df2, percentile=100.0, output_dir=plotpath, 
                       output_name='nregions_per_cell' + paramstring,
                       df1_name=DATASET1_NAME, df2_name=DATASET2_NAME)                 
plot_cnv_per_cell(df1, df2, percentile=100.0, output_dir=plotpath, 
                  output_name='cnv_per_cell' + paramstring, 
                  df1_name=DATASET1_NAME, df2_name=DATASET2_NAME)

df1 = read_data(os.path.join(resultpath1, 'genemetrics_all.csv'))
df2 = read_data(os.path.join(resultpath2, 'genemetrics_all.csv'))
plot_expression_stats(df1, df2, percentile=99.0, output_dir=plotpath, 
                      output_name='genemetrics_all', 
                      df1_name=DATASET1_NAME, df2_name=DATASET2_NAME)

df1 = read_data(os.path.join(resultpath1, 'genemetrics_malignant.csv'))
df2 = read_data(os.path.join(resultpath2, 'genemetrics_malignant.csv'))
plot_expression_stats(df1, df2, percentile=99.0, output_dir=plotpath, 
                      output_name='genemetrics_malignant',
                      df1_name=DATASET1_NAME, df2_name=DATASET2_NAME)

df1 = read_data(os.path.join(resultpath1, 'gene_corr_all.csv'))
df2 = read_data(os.path.join(resultpath2, 'gene_corr_all.csv'))
plot_gene2gene_corr(df1, df2, output_dir=plotpath, 
                    output_name='gene_corr_all',
                    df1_name=DATASET1_NAME, df2_name=DATASET2_NAME)

df1 = read_data(os.path.join(resultpath1, 'gene_corr_malignant.csv'))
df2 = read_data(os.path.join(resultpath2, 'gene_corr_malignant.csv'))
plot_gene2gene_corr(df1, df2, output_dir=plotpath, 
                    output_name='gene_corr_malignant',
                    df1_name=DATASET1_NAME, df2_name=DATASET2_NAME)

df1 = read_data(os.path.join(resultpath1, 'cellmetrics_all.csv'))
df2 = read_data(os.path.join(resultpath2, 'cellmetrics_all.csv'))
plot_cell_counts(df1, df2, output_dir=plotpath, 
                 output_name='cellmetrics_all',
                 df1_name=DATASET1_NAME, df2_name=DATASET2_NAME)

df1 = read_data(os.path.join(resultpath1, 'cellmetrics_malignant.csv'))
df2 = read_data(os.path.join(resultpath2, 'cellmetrics_malignant.csv'))
plot_cell_counts(df1, df2, output_dir=plotpath, 
                 output_name='cellmetrics_malignant',
                 df1_name=DATASET1_NAME, df2_name=DATASET2_NAME)


df1 = read_data(os.path.join(resultpath1, 'cell_corr_all.csv'))
df2 = read_data(os.path.join(resultpath2, 'cell_corr_all.csv'))
plot_cell2cell_corr(df1, df2, output_dir=plotpath, 
                    output_name='cell_corr_all',
                    df1_name=DATASET1_NAME, df2_name=DATASET2_NAME)


df1 = read_data(os.path.join(resultpath1, 'cell_corr_malignant.csv'))
df2 = read_data(os.path.join(resultpath2, 'cell_corr_malignant.csv'))
plot_cell2cell_corr(df1, df2, output_dir=plotpath, 
                    output_name='cell_corr_malignant', 
                    df1_name=DATASET1_NAME, df2_name=DATASET2_NAME)




plt.close()



In [8]:
#cnv summary plot
df_cnv1 = plot_utils.read_data(os.path.join(resultpath1, 'cnv_regions' + paramstring + '.csv'))
df_cnv2 = plot_utils.read_data(os.path.join(resultpath2, 'cnv_regions' + paramstring + '.csv'))
df_subclonal1 = read_data(os.path.join(resultpath1, 'cnvmetrics_per_subclonal' + paramstring + '.csv'))
df_subclonal2 = read_data(os.path.join(resultpath2, 'cnvmetrics_per_subclonal' + paramstring + '.csv'))

plot_summary_cnv(df_cnv1, df_cnv2, df_subclonal1, df_subclonal2, 
              percentile=100, output_dir=plotpath, output_name='cnvmetrics_summary' + paramstring,
              df1_name=DATASET1_NAME, df2_name=DATASET2_NAME)



In [9]:
#normal summary plot
df_gene_all1 = read_data(os.path.join(resultpath1, 'genemetrics_all.csv'))
df_gene_all2 = read_data(os.path.join(resultpath2, 'genemetrics_all.csv'))
df_gene_mal1 = read_data(os.path.join(resultpath1, 'genemetrics_malignant.csv'))
df_gene_mal2 = read_data(os.path.join(resultpath2, 'genemetrics_malignant.csv'))
df_counts_all1 = read_data(os.path.join(resultpath1, 'cellmetrics_all.csv'))
df_counts_all2 = read_data(os.path.join(resultpath2, 'cellmetrics_all.csv'))
df_counts_mal1 = read_data(os.path.join(resultpath1, 'cellmetrics_malignant.csv'))
df_counts_mal2 = read_data(os.path.join(resultpath2, 'cellmetrics_malignant.csv'))
df_corr_all1 = read_data(os.path.join(resultpath1, 'cell_corr_all.csv'))
df_corr_all2 = read_data(os.path.join(resultpath2, 'cell_corr_all.csv'))
df_corr_mal1 = read_data(os.path.join(resultpath1, 'cell_corr_malignant.csv'))
df_corr_mal2 = read_data(os.path.join(resultpath2, 'cell_corr_malignant.csv'))


plot_summary_normal(df_gene_all1, df_gene_all2, df_gene_mal1, df_gene_mal2,
                  df_counts_all1, df_counts_all2, df_counts_mal1, df_counts_mal2,
                  df_corr_all1, df_corr_all2, df_corr_mal1, df_corr_mal2,
              percentile=100, output_dir=plotpath, output_name='normalmetrics_summary',
              df1_name=DATASET1_NAME, df2_name=DATASET2_NAME)


In [10]:
#location plots
df1 = read_data(os.path.join(resultpath1, 'cnv_regions_unique' + paramstring + '.csv'))
df2 = read_data(os.path.join(resultpath2, 'cnv_regions_unique' + paramstring + '.csv'))
single_plots.plot_chr_locations(df1, output_dir=plotpath, output_name='chr_locations'  + paramstring + '_' + DATASET1_NAME)
single_plots.plot_chr_locations(df2, output_dir=plotpath, output_name='chr_locations'  + paramstring +'_' + DATASET2_NAME)

single_plots.plot_chr_location_and_size_all(df1, output_dir=plotpath, output_name= 'chr_locations_and_size_all' + paramstring + '_' + DATASET1_NAME )
single_plots.plot_chr_location_and_size_all(df2, output_dir=plotpath, output_name= 'chr_locations_and_size_all' + paramstring + '_' + DATASET2_NAME )
single_plots.plot_chr_location_and_size_gl(df1, output_dir=plotpath, output_name= 'chr_locations_and_size_gl' + paramstring + '_' + DATASET1_NAME)
single_plots.plot_chr_location_and_size_gl(df2, output_dir=plotpath, output_name= 'chr_locations_and_size_gl' + paramstring + '_' + DATASET2_NAME)

plt.close()


/notebook/Z/benchmark/plot/single_plots.py:447: RuntimeWarning: invalid value encountered in divide
  bins_chr_n[i, :] /= float(sum_)
/usr/local/lib/python3.8/dist-packages/seaborn/axisgrid.py:118: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
/usr/local/lib/python3.8/dist-packages/seaborn/axisgrid.py:118: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
/usr/local/lib/python3.8/dist-packages/seaborn/axisgrid.py:118: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
/usr/local/lib/python3.8/dist-packages/seaborn/axisgrid.py:118: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
